# Dose Response Study

Simulating dose response experiments.

We begin by importing required packages.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.optimize as opt
import statistics
from scipy import stats
import os
import re
import ptitprince as pt #rain plots
from datetime import date

import libraries.disturbances as dt
import libraries.normalization as nrm
import libraries.dose_response as dr
import libraries.utilities as util

In [3]:
data_dir = "generated-data/dose-response/"

In [4]:
def full_dose_response_evaluation(plate_types_location, error_types, e_from=1, e_to=100, e_step=5,
                                  compounds = 48, concentrations = 6, replicates = 1, dilution = 18,
                                  error_nl = 0.055,
                                  lose_rows_from=1, lose_rows_to=2, today = (date.today()).strftime("%Y%m%d")+"-test",
                                  data_dir = data_dir):
    ## Results
    absolute_ic50_data_f=open(data_dir + 'absolute_ic50_data-'+str(compounds)+'-'+str(concentrations)+'-dil'+str(dilution)
                              +'-'+str(replicates)+'-'+str(error_nl)+'-'+today+'.csv','a')
    relative_ic50_data_f=open(data_dir + 'relative_ic50_data-'+str(compounds)+'-'+str(concentrations)+'-dil'
                              +str(dilution)+'-'+str(replicates)+'-'+str(error_nl)+'-'+today+'.csv','a')
    residuals_f=open(data_dir + 'residuals-'+str(compounds)+'-'+str(concentrations)+'-dil'+str(dilution)
                     +'-'+str(replicates)+'-'+str(error_nl)+'-'+today+'.csv','a')

    for current_e in range(e_from,e_to,e_step):
        print("\nTesting compounds with e in range ",current_e,"-",(current_e+e_step),":")

        # Create new curves/compounds

        params = dr.generate_compound_curves(compounds,concentrations,dilution,current_e)

        df_params = pd.DataFrame.from_dict(params)
        df_params.set_index('compound',inplace=True)
        df_params['abs IC50'] = dr.IC50(df_params['b'],df_params['c'],df_params['d'],df_params['e'])

        compounds_array = df_params.index.to_numpy()

        # The same compounds/concentrations will be used in every plate
        plate_content = dr.generate_plate_content(dose_response_params=params, replicates=replicates)

        for plate_type in plate_types_location:
            print("Using ",plate_type['type']," layouts...")
            layout_dir = plate_type['dir']
            layouts = os.listdir(layout_dir)

            for layout_file in layouts:
                match = re.search(plate_type['regex'],layout_file)

                # Skip this file if it doesn't match the regular expression in plate_type['regex']
                if match == None:
                    continue

                layout_file_array = np.full(len(compounds_array),layout_file)
                e_array = np.full(len(compounds_array),current_e)

                for et in error_types:
                    error_type_array = np.full(len(compounds_array),et['type'])
                    error_array = np.full(len(compounds_array),et['error'])    
                    for lost_rows in range(1,2): #max 8
                        limits = [#{'from':0, 'to':lost_rows},
                                  {'from':16-lost_rows,'to':16}]
                        for limit in limits:
                            function(compounds_array,layout_file_array,error_type_array,error_array,e_array,lost_rows,
                                     layout_file,
                                     plate_content,et,current_e,df_params,
                                     plate_type,compounds,concentrations,replicates,
                                     limit,absolute_ic50_data_f,relative_ic50_data_f,residuals_f)
    ## Close all files
    absolute_ic50_data_f.close()
    relative_ic50_data_f.close()
    residuals_f.close()

def function(compounds_array,layout_file_array,error_type_array,error_array,e_array,lost_rows,
             layout_file,plate_content,et,current_e,df_params,
             plate_type,compounds,concentrations,replicates,
             limit,absolute_ic50_data_f,relative_ic50_data_f,residuals_f,expected_noise=0.01):
    layout_dir = plate_type['dir']
    r_lost_array = np.full(len(compounds_array),lost_rows)

    fitTable_new = dr.plate_curves_after_error(layout_dir,layout_file,plate_content,expected_noise,
                                               et['error_function'],et['error'],plate_type['error_correction'],
                                               lose_from_row=limit['from'],lose_to_row=limit['to'],
                                               plate_type  = plate_type,
                                               compounds = compounds, concentrations = concentrations, replicates = replicates
                                              )
    obtained_absolute_ic50 = dr.IC50(fitTable_new['b'],fitTable_new['c'],fitTable_new['d'],fitTable_new['e'])

    res_array = np.concatenate(fitTable_new['residuals'].to_numpy())
    true_res_array = np.concatenate(fitTable_new['true_residuals'].to_numpy())
    res_size = len(res_array)

    plate_residuals = np.vstack([np.full(res_size,layout_file), np.full(res_size,et['type']), np.full(res_size,et['error']),
                                 np.full(res_size,current_e), np.full(res_size,lost_rows), res_array, true_res_array])

    np.savetxt(residuals_f, plate_residuals.T, delimiter=",",fmt='%s')

    plate_rel_ic50 = np.absolute(np.subtract(np.log10(df_params['e']), 
                                             np.log10(fitTable_new['e'])))
    plate_abs_ic50 = np.absolute(np.subtract(np.log10(df_params['abs IC50']), 
                                             np.log10(obtained_absolute_ic50)))
    # Includes curve info
    rel_results = np.vstack([layout_file_array, compounds_array, plate_rel_ic50.to_numpy(), error_type_array, 
                             error_array, e_array, r_lost_array, fitTable_new['r2_score'],df_params['b'],df_params['c'],
                             df_params['d'],df_params['e'],fitTable_new['b'],fitTable_new['c'],fitTable_new['d'],fitTable_new['e']])
    abs_results = np.vstack([layout_file_array, compounds_array, plate_abs_ic50.to_numpy(), error_type_array, 
                             error_array, e_array, r_lost_array, fitTable_new['r2_score'],df_params['b'],df_params['c'],
                             df_params['d'],df_params['e'],fitTable_new['b'],fitTable_new['c'],fitTable_new['d'],fitTable_new['e']])

    # Save results
    np.savetxt(absolute_ic50_data_f, abs_results.T, delimiter=",",fmt='%s')
    np.savetxt(relative_ic50_data_f, rel_results.T, delimiter=",",fmt='%s')

## Data Generation

Here you can specify the particulars of the experiment.

In [6]:
## Simulation parameters
## Ensure the appropriate PLAID layouts exist under "layouts/compounds_PLAID_layouts"

concentrations_list = [6, 8, 12]
concentrations = 8 #Possible to use 4, 6, 8
replicates_list = [1, 2, 3]
replicates = 1

# Total number of compounds that fit in a 384 well-plate after leaving the outer wells empty
# 
compounds = (14*22-20)//(concentrations*replicates)

if concentrations == 4:
    dilution = 15
elif concentrations == 6:
    dilution = 18
elif concentrations == 8:
    dilution = 8
elif concentrations == 12:
    dilution = 4
else:
    print("Please specify the dilution factor")

id_text = "manuscript"

#expected_noise = 0.01
#error_nl_levels = [0.
#error_nl = 0.4

#my_min_dist = 0

In [7]:
e_from=1
e_to=5##100
#e_to=100
e_step=5
today = (date.today()).strftime("%Y%m%d")+"-"

In [8]:
#error_types = [{'type':"right-exp", 'error_function':dt.add_exponential_errors_to_right_columns,'error_correction':nrm.normalize_plate_column_effect,'error':error_nl}]
#error_types = [{'type':"right-half", 'error_function':dt.add_errors_to_right_columns_half,'error_correction':nrm.normalize_plate_nearest_control,'error':error_nl}]
#error_types = [{'type':"diagonal", 'error_function':dt.add_diagonal_errors,'error_correction':nrm.normalize_plate_nearest_control,'error':error_nl}]
#error_types = [{'type':"bowl-nl", 'error_function':dt.add_bowlshaped_errors_nl,'error_correction':nrm.normalize_plate_nearest_control,'error':error_nl}]
               #{'type':"bowl", 'error_function':dt.add_bowlshaped_errors,'error_correction':nrm.normalize_plate_nearest_control,'error':error_l}]
               #{'type':"left-nl", 'error_function':dt.add_errors_to_left_columns,'error_correction':nrm.normalize_plate_column_effect,'error':error_nl},
               #{'type':"left", 'error_function':dt.add_linear_errors_to_left_columns,'error_correction':nrm.normalize_plate_column_effect,'error':error},
               #{'type':"right", 'error_function':dt.add_linear_errors_to_right_columns,'error_correction':nrm.normalize_plate_column_effect,'error':error},
               #{'type':"top-nl", 'error_function':dt.add_errors_to_upper_rows,'error_correction':nrm.normalize_plate_row_effect,'error':error_nl},
               #{'type':"top", 'error_function':dt.add_linear_errors_to_upper_rows,'error_correction':nrm.normalize_plate_row_effect,'error':error},
               #{'type':"bottom", 'error_function':dt.add_linear_errors_to_lower_rows,'error_correction':nrm.normalize_plate_row_effect,'error':error},
               #{'type':"left-exp", 'error_function':dt.add_exponential_errors_to_left_columns,'error_correction':nrm.normalize_plate_column_effect,'error':error_nl},
               #{'type':"top-exp", 'error_function':dt.add_exponential_errors_to_upper_rows,'error_correction':nrm.normalize_plate_row_effect,'error':error_nl}]
#{'type':,error_function':,'error_correction':}

id_text_error_level_type_list = [ 
      ('right-half-neg-control-log-new-reg', 0.2,  [{'type':"right-half", 'error_function':dt.add_errors_to_right_columns_half,'error_correction':nrm.normalize_plate_nearest_control,'error':0.2}]),
      ('log-neg-control-new-reg',            0.4,  [{'type':"right-half", 'error_function':dt.add_errors_to_right_columns_half,'error_correction':nrm.normalize_plate_nearest_control,'error':0.4}]),
      ('right-half-neg-control-log-new-reg', 0.4,  [{'type':"right-half", 'error_function':dt.add_errors_to_right_columns_half,'error_correction':nrm.normalize_plate_nearest_control,'error':0.4}]),
      ('curve_info-new-reg',                 0.055,[{'type':"bowl-nl", 'error_function':dt.add_bowlshaped_errors_nl,'error_correction':nrm.normalize_plate_nearest_control,'error':0.055}]),
      ('bowl-neg-control-new-reg',           0.055,[{'type':"bowl-nl", 'error_function':dt.add_bowlshaped_errors_nl,'error_correction':nrm.normalize_plate_nearest_control,'error':0.055}]),
      ('curve_info-new-reg',                 0.085,[{'type':"bowl-nl", 'error_function':dt.add_bowlshaped_errors_nl,'error_correction':nrm.normalize_plate_nearest_control,'error':0.085}]),
      ('bowl-neg-control-new-reg',           0.085,[{'type':"bowl-nl", 'error_function':dt.add_bowlshaped_errors_nl,'error_correction':nrm.normalize_plate_nearest_control,'error':0.085}])
    ]

Now we place them on a plate, add a small random noise, apply plate effects error and correct them.

In [10]:
r = 0
r_max = len(concentrations_list) * len(replicates_list) * len(id_text_error_level_type_list)
for concentrations in concentrations_list:
     for replicates in replicates_list:

        compounds = (14*22-20)//(concentrations*replicates)
        
        if concentrations == 4:
            dilution = 15
        elif concentrations == 6:
            dilution = 18
        elif concentrations == 8:
            dilution = 8
        elif concentrations == 12:
            dilution = 4
        else:
            print("Please specify the dilution factor")
        
        for (id_text, error_nl, error_types) in id_text_error_level_type_list:
            print("Results are being stored in files which include the name: ",str(compounds)+'-'+str(concentrations)+'-dil'+str(dilution)+'-'+str(replicates)+'-'+str(error_nl)+'-'+today+id_text)
            plate_types_location = [{'type':'COMPD', 'dir':'layouts/compounds_COMPD_layouts/', 'regex':'plate_layout_(.*)'+str(compounds)+'-'+str(concentrations)+'-'+str(replicates)+'_(0*)(.+?).npy','error_correction':nrm.normalize_plate_lowess_2d},
                                    {'type':'PLAID', 'dir':'layouts/compounds_PLAID_layouts/', 'regex':'plate_layout_(.*)'+str(compounds)+'-'+str(concentrations)+'-'+str(replicates)+'_(0*)(.+?).npy','error_correction':nrm.normalize_plate_lowess_2d},
                                    {'type':'RANDOM','dir':'layouts/compounds_manual_layouts/','regex':'plate_layout_rand_(.+?).npy',                                                                  'error_correction':nrm.normalize_plate_lowess_2d}
                                   ]
            
            full_dose_response_evaluation(plate_types_location, error_types,
                                          compounds = compounds, concentrations = concentrations,
                                          replicates = replicates, dilution = dilution,
                                          error_nl = error_nl, today = today + id_text)
            r += 1
            print(r,'out of',r_max)


Results are being stored in files which include the name:  48-6-dil18-1-0.2-20250706-right-half-neg-control-log-new-reg

Testing compounds with e in range  1 - 6 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  6 - 11 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  11 - 16 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  16 - 21 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  21 - 26 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  26 - 31 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  31 - 36 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in 

/Users/ramgi410/Documents-Local/eclipse_workspace/compd_evaluation/libraries/dose_response.py:20: RuntimeWarning: overflow encountered in exp
  result = c + (d-c)/(1+np.exp(b*(np.log(x)-np.log(e))))


Using  RANDOM  layouts...

Testing compounds with e in range  41 - 46 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  46 - 51 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  51 - 56 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  56 - 61 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  61 - 66 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  66 - 71 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  71 - 76 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  76 - 81 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts..

/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_lsq/common.py:115: RuntimeWarning: divide by zero encountered in divide
  phi_prime = -np.sum(suf ** 2 / denom**3) / p_norm
/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_lsq/common.py:115: RuntimeWarning: divide by zero encountered in divide
  phi_prime = -np.sum(suf ** 2 / denom**3) / p_norm
/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_lsq/common.py:115: RuntimeWarning: divide by zero encountered in divide
  phi_prime = -np.sum(suf ** 2 / denom**3) / p_norm



Testing compounds with e in range  16 - 21 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  21 - 26 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  26 - 31 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  31 - 36 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  36 - 41 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  41 - 46 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  46 - 51 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  51 - 56 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with 

...


Testing compounds with e in range  81 - 86 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...


/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_lsq/common.py:115: RuntimeWarning: divide by zero encountered in divide
  phi_prime = -np.sum(suf ** 2 / denom**3) / p_norm



Testing compounds with e in range  86 - 91 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  91 - 96 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  96 - 101 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...
24 out of 63
Results are being stored in files which include the name:  36-8-dil8-1-0.055-20250706-curve_info-new-reg

Testing compounds with e in range  1 - 6 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  6 - 11 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  11 - 16 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  16 - 21 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in r

/Users/ramgi410/Documents-Local/eclipse_workspace/compd_evaluation/libraries/dose_response.py:20: RuntimeWarning: overflow encountered in exp
  result = c + (d-c)/(1+np.exp(b*(np.log(x)-np.log(e))))


Using  RANDOM  layouts...

Testing compounds with e in range  96 - 101 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...
28 out of 63
Results are being stored in files which include the name:  18-8-dil8-2-0.2-20250706-right-half-neg-control-log-new-reg

Testing compounds with e in range  1 - 6 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  6 - 11 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  11 - 16 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  16 - 21 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  21 - 26 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  layouts...

Testing compounds with e in range  26 - 31 :
Using  COMPD  layouts...
Using  PLAID  layouts...
Using  RANDOM  l

In [11]:
print("\nDone! :-)")


Done! :-)


In [12]:
# for test purposes
plate = np.array(
[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,    0,   0,   0,   0,   0,   0],
 [  0,  59, 125, 145,  45,  93,  71,   9, 145, 123,  85,  57,  56,  34,   2, 145,  46,  76,   22, 116,   4,  28, 145,   0],
 [  0,   3,  65, 111, 141,  17,  23,  79, 119,  41,  33,  89, 126, 104,   8,  16,  98, 140,   96, 114, 138,  86,  58,   0],
 [  0, 105,  19,  73,  97,  53, 127,  61, 139,  13, 137,  27,  62,  38, 112,  70,  50, 118,  145,  84,  12,   6,  36,   0],
 [  0, 145,  15,  95,   7,  81, 115,  75, 109,  99, 145, 107,  92,  78,  20, 134, 106,  40,   88,  18,  54,  24, 132,   0],
 [  0,  31,  37,  69, 103, 145, 135,  91,  67,  83,  21, 117, 142,  80, 145,  64,  94, 130,   74,  26,  44,  68, 108,   0],
 [  0,  29, 121,  35,  43, 129, 101,  39,   1,  11,  63,  49,  72, 102, 128,  48,  32,  82,  120, 100, 145,  66, 122,   0],
 [  0,  55,  87,  25,   5, 143, 113,  51,  47, 133, 131,  77,  30, 110,  14,  42, 144, 136,   52,  10, 124,  60,  90,   0],
 [  0,  30, 145, 116,  66,  74,  56,  20,  14,   4,  34, 145,  45,  11,  91,  21,  97, 145,   69, 129,  31,  27,  57,   0],
 [  0,   2, 132,   8,  40,  72, 138, 145,  92,  42, 112, 100,  35, 119,  67,  61,  73, 109,   93,  23, 137,   5,  55,   0],
 [  0, 104,  96,  50, 110,  16,  48, 142, 106,  76,   6,  24, 133, 141,  49, 105,  13,   9,   37,  33,  83, 115,  25,   0],
 [  0,  36,  70,  88, 145,  98,  80,  64, 118,  10,  52, 130, 125,  77,  43, 131,   1, 143,   65, 145,  17,  53, 127,   0],
 [  0, 124,  62,  46, 120,  22,  94,  38, 136,  44, 140,  84,  29, 145,   7, 113,  19, 117,   81,  75, 111,  99,  79,   0],
 [  0,  86,  28, 134, 144,  12, 128,  18, 102, 145, 108, 126,  87,   3, 103,  63, 101,  39,   71,  95, 139, 107, 121,   0],
 [  0,  58,  90,  32,  82,  78,  54,  68, 114, 122,  60,  26,  59,  85, 123, 135, 145,  47,   15,  51,  41, 145,  89,   0],
 [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,    0,   0,   0,   0,   0,   0]])